### 下載檔案

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("joebeachcapital/30000-spotify-songs")

print("Path to dataset files:", path)

Path to dataset files: /Users/angus3938/.cache/kagglehub/datasets/joebeachcapital/30000-spotify-songs/versions/2


In [3]:
import pandas as pd

view = pd.read_csv(f"{path}/spotify_songs.csv")  # 根據實際檔名修改

---

# version 1
# completed function

In [1]:
# ===================================================================
# 1. Core Helper Functions
# ===================================================================

def _attempt_convert_type(value_str):
    """Automatically convert string values to appropriate data types"""
    # Handle empty strings
    if not value_str or value_str.strip() == '':
        return None
    
    value_str = value_str.strip()
    
    # Handle quoted strings
    if value_str.startswith('"') and value_str.endswith('"'):
        return value_str[1:-1]
    
    # Try converting to integer
    try:
        return int(value_str)
    except ValueError:
        pass
    
    # Try converting to float
    try:
        return float(value_str)
    except ValueError:
        pass
    
    # Handle boolean values
    if value_str.lower() in ['true', 'false']:
        return value_str.lower() == 'true'
    
    # Return original string
    return value_str

def parse_csv_line(line, separator=",", quote_char='"'):
    """
    Parse a CSV line while properly handling separators inside quotes
    """
    data_per_row = []
    current_value = ""
    in_quotes = False
    
    for char in line:
        if char == quote_char:
            in_quotes = not in_quotes
        elif char == separator and not in_quotes:
            data_per_row.append(current_value)
            current_value = ""
        else:
            current_value += char
    
    # Add the last value
    data_per_row.append(current_value)
    return data_per_row


# ===================================================================
# 2. Core DataFrame & GroupBy Classes
# ===================================================================

class DataFrame:
    def __init__(self, data_dict):
        """
        Initialize DataFrame with a dictionary of columns
        data_dict: {'column1': [values], 'column2': [values]}
        """
        if not data_dict:
            raise ValueError("Cannot create DataFrame from empty data")
        
        # Ensure all columns have consistent row count
        lengths = [len(v) for v in data_dict.values()]
        if len(set(lengths)) > 1:
            mismatched = {k: len(v) for k, v in data_dict.items()}
            raise ValueError(f"All columns must have the same length. Mismatched lengths: {mismatched}")
        
        self.data = data_dict
        self.columns = list(data_dict.keys())
        self.row_count = lengths[0] if lengths else 0
    
    def __repr__(self):
        """Return a formatted string representation of the DataFrame (similar to pandas)"""
        if self.row_count == 0:
            return "Empty DataFrame"
        
        # Compute column widths
        col_widths = {}
        for col in self.columns:
            max_width = len(col)
            for val in self.data[col][:10]:  # Only check the first 10 rows
                max_width = max(max_width, len(str(val)))
            col_widths[col] = min(max_width, 20)  # Limit width
        
        result = []
        
        # Header
        header = " | ".join(col.ljust(col_widths[col]) for col in self.columns)
        result.append(header)
        result.append("-" * len(header))
        
        # Data rows (up to 10)
        display_rows = min(10, self.row_count)
        for i in range(display_rows):
            row = []
            for col in self.columns:
                val = str(self.data[col][i])
                if len(val) > col_widths[col]:
                    val = val[:col_widths[col]-3] + "..."
                row.append(val.ljust(col_widths[col]))
            result.append(" | ".join(row))
        
        if self.row_count > 10:
            result.append(f"\n... {self.row_count - 10} more rows")
        
        result.append(f"\nShape: ({self.row_count} rows, {len(self.columns)} columns)")
        
        return "\n".join(result)
    
    def __getitem__(self, key):
        """
        Support column selection, multiple column selection, and boolean indexing
        """
        if isinstance(key, str):
            return self.data[key]  # Single column
        
        elif isinstance(key, list) and all(isinstance(k, str) for k in key):
            return DataFrame({col: self.data[col] for col in key})  # Subset of columns
        
        elif isinstance(key, list) and all(isinstance(k, bool) for k in key):
            # Boolean filter
            if len(key) != self.row_count:
                raise ValueError("Boolean index length mismatch")
            
            new_data = {col: [] for col in self.columns}
            for i, keep in enumerate(key):
                if keep:
                    for col in self.columns:
                        new_data[col].append(self.data[col][i])
            
            return DataFrame(new_data)
        
        else:
            raise TypeError(f"Invalid indexing type: {type(key)}")
    
    def __len__(self):
        return self.row_count

    # ==================== 1. Filtering ====================
    
    def filter(self, condition):
        """
        Filter rows by a condition
        
        parameters:
            condition: function or boolean list
                - function: lambda row: row['Age'] > 18
                - list: [True, False, True, ...]
        
        Returns:
            DataFrame: filtered result
        """
        if callable(condition):
            keep_rows = []
            for i in range(self.row_count):
                row = {col: self.data[col][i] for col in self.columns}
                keep_rows.append(condition(row))
            return self[keep_rows]
        
        elif isinstance(condition, list) and all(isinstance(k, bool) for k in condition):
            return self[condition]
        
        else:
            raise TypeError("Condition must be callable or a list of booleans")
    
    # ==================== 2. Projection ====================
    
    def select(self, columns):
        """
        Select one or more columns
        
        Parameters:
            columns: str or list
        
        Returns:
            DataFrame
        """
        if isinstance(columns, str):
            columns = [columns]
        
        return self[columns]
    
    # ==================== 3. GroupBy ====================
    
    def groupby(self, by):
        """
        Group rows by one or more columns
        
        Parameters:
            by: str or list
        
        Returns:
            GroupBy object
        """
        if isinstance(by, str):
            by = [by]
        
        return GroupBy(self, by)
    
    # ==================== 4. Join ====================
    
    def join(self, other, left_on, right_on, how='inner'):
        """
        Join with another DataFrame
        
        parameters:
            other: DataFrame - the DataFrame to join with
            left_on: str - key column in the left DataFrame
            right_on: str - key column in the right DataFrame
            how: str - join type ('inner', 'left', 'right', 'outer')
        
        returns:
            DataFrame: join result
        """
        # Build index for right DataFrame
        right_index = {}
        for i, val in enumerate(other.data[right_on]):
            right_index.setdefault(val, []).append(i)
        
        # Initialize result container
        result_data = {col: [] for col in self.columns}
        for col in other.columns:
            if col != right_on:  # Avoid duplicate key columns
                result_data[f"{col}_right"] = []
        
        matched_right_indices = set()
        
        # Iterate through left DataFrame rows
        for i in range(self.row_count):
            left_key = self.data[left_on][i]
            
            if left_key in right_index:
                for right_i in right_index[left_key]:
                    matched_right_indices.add(right_i)
                    
                    # Add left columns
                    for col in self.columns:
                        result_data[col].append(self.data[col][i])
                    
                    # Add right columns
                    for col in other.columns:
                        if col != right_on:
                            result_data[f"{col}_right"].append(other.data[col][right_i])
            
            elif how in ['left', 'outer']:
                # Keep unmatched left rows
                for col in self.columns:
                    result_data[col].append(self.data[col][i])
                for col in other.columns:
                    if col != right_on:
                        result_data[f"{col}_right"].append(None)
        
        # Add unmatched right rows
        if how in ['right', 'outer']:
            for right_i in range(other.row_count):
                if right_i not in matched_right_indices:
                    for col in self.columns:
                        result_data[col].append(None)
                    for col in other.columns:
                        if col != right_on:
                            result_data[f"{col}_right"].append(other.data[col][right_i])
        
        return DataFrame(result_data)
    
    # ==================== Helper Methods ====================
    
    def head(self, n=5):
        """Return the first n rows"""
        return DataFrame({col: self.data[col][:n] for col in self.columns})
    
    def tail(self, n=5):
        """Return the last n rows"""
        return DataFrame({col: self.data[col][-n:] for col in self.columns})
    
    def shape(self):
        """Return (rows, columns)"""
        return (self.row_count, len(self.columns))
    
    def info(self):
        """Print DataFrame metadata"""
        print("DataFrame Info:")
        print(f"Rows: {self.row_count}")
        print(f"Columns: {len(self.columns)}\n")
        print("Column Names and Types:")
        for col in self.columns:
            sample_val = self.data[col][0] if self.row_count > 0 else None
            print(f"  {col}: {type(sample_val).__name__}")
    
    @classmethod
    def from_csv(cls, filepath, separator=",", quote_char='"'):
        """Create a DataFrame from a CSV file (loaded into memory)"""
        data_dict = load_csv_advanced(filepath, separator, quote_char)
        if data_dict is None:
            raise ValueError(f"Failed to load CSV from {filepath}")
        return cls(data_dict)


# ==================== GroupBy Class ====================

class GroupBy:
    def __init__(self, dataframe, by):
        """
        GroupBy object representing grouped data
        """
        self.df = dataframe
        self.by = by
        self.groups = self._create_groups()
    
    def _create_groups(self):
        """Build grouping index"""
        groups = {}
        
        for i in range(self.df.row_count):
            key_values = tuple(self.df.data[col][i] for col in self.by)
            groups.setdefault(key_values, []).append(i)
        
        return groups
    
    def aggregate(self, agg_dict):
        """
        Perform aggregation
        
        parameters:
            agg_dict: dict - {column_name: aggregation_function}
                supported: 'sum', 'mean', 'max', 'min', 'count', 'std'
        
        returns:
            DataFrame
        """
        result_data = {col: [] for col in self.by}
        
        for col, func in agg_dict.items():
            result_data[f"{col}_{func}"] = []
        
        for key_values, indices in self.groups.items():
            # Add group keys
            for i, col in enumerate(self.by):
                result_data[col].append(key_values[i])
            
            # Compute aggregations
            for col, func_name in agg_dict.items():
                values = [self.df.data[col][i] for i in indices]
                values = [v for v in values if v is not None]
                
                if not values:
                    result = None
                else:
                    result = self._apply_aggregation(values, func_name)
                
                result_data[f"{col}_{func_name}"].append(result)
        
        return DataFrame(result_data)
    
    def _apply_aggregation(self, values, func_name):
        """Apply aggregation function"""
        if func_name == 'sum':
            return sum(values)
        elif func_name == 'mean':
            return sum(values) / len(values)
        elif func_name == 'max':
            return max(values)
        elif func_name == 'min':
            return min(values)
        elif func_name == 'count':
            return len(values)
        elif func_name == 'std':
            if len(values) < 1:
                return None
            mean = sum(values) / len(values)
            variance = sum((x - mean) ** 2 for x in values) / len(values)
            return variance ** 0.5
        else:
            raise ValueError(f"Unknown aggregation function: {func_name}")
    
    def size(self):
        """Return the size of each group"""
        result_data = {col: [] for col in self.by}
        result_data['size'] = []
        
        for key_values, indices in self.groups.items():
            for i, col in enumerate(self.by):
                result_data[col].append(key_values[i])
            result_data['size'].append(len(indices))

        return DataFrame(result_data)
  

# ===================================================================
# 3. Advanced CSV Loader (In-Memory)
# ===================================================================

def load_csv_advanced(filePath, separator=",", quote_char='"'):
    """
    Advanced CSV loader that properly handles quoted fields
    Example: "Smith, John",25,USA
    Loads the entire file into memory.
    """
    data_dict = {}
    header = []
    
    try:
        with open(filePath, "r", encoding="utf-8") as f:
            # Read header row
            header_line = f.readline().strip()
            header = parse_csv_line(header_line, separator, quote_char)
            header = [col.strip().strip(quote_char) for col in header]
            
            data_dict = {col_name: [] for col_name in header}
            
            # Read data rows
            for line_number, line in enumerate(f, start=2):
                cleaned_line = line.strip()
                if not cleaned_line:
                    continue
                
                values = parse_csv_line(cleaned_line, separator, quote_char)
                
                if len(values) == len(header):
                    for i, col_name in enumerate(header):
                        converted_value = _attempt_convert_type(values[i])
                        data_dict[col_name].append(converted_value)
                else:
                    print(f"Warning: Line {line_number} column mismatch (Expected {len(header)}, got {len(values)})")
        
        return data_dict
        
    except FileNotFoundError:
        print(f"Error: File not found at {filePath}")
        return None
    except Exception as e:
        print(f"Error reading CSV: {e}")
        import traceback
        traceback.print_exc()
        return None


# ===================================================================
# 4. Chunked/Batch Processing
# ===================================================================

class CSVChunkReader:
    """
    Chunk-based CSV reader.
    Reads large CSV files in small batches (memory-efficient).
    Returns each chunk as a DataFrame.
    """
    
    def __init__(self, filepath, separator=",", quote_char='"', chunk_size=1000):
        """
        Initialize chunk reader
        
        parameters:
            filepath: str - path to the CSV file
            separator: str - field separator
            quote_char: str - quote character
            chunk_size: int - number of rows per chunk
        """
        self.filepath = filepath
        self.separator = separator
        self.quote_char = quote_char
        self.chunk_size = chunk_size
        self.header = None
        self.total_rows_read = 0
    
    def __iter__(self):
        return self.read_chunks()
    
    def read_chunks(self):
        """
        Generator that yields DataFrame chunks
        
        Yields:
            DataFrame - batch of rows
        """
        try:
            with open(self.filepath, 'r', encoding='utf-8') as f:
                header_line = f.readline().strip()
                if not header_line:
                    raise ValueError("File is empty")
                
                self.header = parse_csv_line(header_line, self.separator, self.quote_char)
                self.header = [col.strip().strip(self.quote_char) for col in self.header]
                
                chunk_data = {col: [] for col in self.header}
                chunk_row_count = 0
                
                for line_number, line in enumerate(f, start=2):
                    cleaned_line = line.strip()
                    if not cleaned_line:
                        continue
                    
                    values = parse_csv_line(cleaned_line, self.separator, self.quote_char)
                    
                    if len(values) == len(self.header):
                        for i, col in enumerate(self.header):
                            converted_value = _attempt_convert_type(values[i])
                            chunk_data[col].append(converted_value)
                        
                        chunk_row_count += 1
                        self.total_rows_read += 1
                        
                        if chunk_row_count >= self.chunk_size:
                            yield DataFrame(chunk_data)
                            chunk_data = {col: [] for col in self.header}
                            chunk_row_count = 0
                    else:
                        print(f"Warning: Line {line_number} column mismatch (Expected {len(self.header)}, got {len(values)})")
                
                if chunk_row_count > 0:
                    yield DataFrame(chunk_data)
        
        except FileNotFoundError:
            print(f"Error: File not found at {self.filepath}")
            return
        except Exception as e:
            print(f"Error reading CSV chunks: {e}")
            import traceback
            traceback.print_exc()
            return


def read_csv_chunks(filepath, separator=",", quote_char='"', chunk_size=1000):
    """
    Convenience function: chunked CSV reader
    
    Example:
        for chunk in read_csv_chunks("large.csv", chunk_size=1000):
            print(chunk.shape())
    """
    return CSVChunkReader(filepath, separator, quote_char, chunk_size)


class ChunkProcessor:
    """
    Utilities for applying operations on CSV chunks.
    Suitable for large datasets.
    """
    
    @staticmethod
    def filter_chunks(chunk_reader, condition):
        """
        Apply row filtering to each chunk
        """
        for chunk in chunk_reader:
            filtered = chunk.filter(condition)
            if len(filtered) > 0:
                yield filtered
    
    @staticmethod
    def aggregate_chunks(chunk_reader, group_by_cols, agg_dict):
        """
        Perform simplified chunk-based aggregation
        (Aggregations are merged at the end)
        """
        if isinstance(group_by_cols, str):
            group_by_cols = [group_by_cols]
        
        accumulated_groups = {}
        
        for chunk in chunk_reader:
            grouped = chunk.groupby(group_by_cols)
            chunk_result = grouped.aggregate(agg_dict)
            
            for i in range(len(chunk_result)):
                group_key = tuple(chunk_result.data[col][i] for col in group_by_cols)
                
                if group_key not in accumulated_groups:
                    accumulated_groups[group_key] = {}
                    for col in group_by_cols:
                        accumulated_groups[group_key][col] = chunk_result.data[col][i]
                    for agg_col in agg_dict.keys():
                        func = agg_dict[agg_col]
                        accumulated_groups[group_key][f"{agg_col}_{func}"] = []
                
                for agg_col in agg_dict.keys():
                    func = agg_dict[agg_col]
                    result_col = f"{agg_col}_{func}"
                    accumulated_groups[group_key][result_col].append(
                        chunk_result.data[result_col][i]
                    )
        
        final_data = {col: [] for col in group_by_cols}
        for agg_col in agg_dict.keys():
            func = agg_dict[agg_col]
            final_data[f"{agg_col}_{func}"] = []
        
        for group_key, group_data in accumulated_groups.items():
            for i, col in enumerate(group_by_cols):
                final_data[col].append(group_key[i])
            
            for agg_col in agg_dict.keys():
                func = agg_dict[agg_col]
                result_col = f"{agg_col}_{func}"
                values = [v for v in group_data[result_col] if v is not None]
                
                if not values:
                    final_value = None
                elif func == 'sum':
                    final_value = sum(values)
                elif func == 'max':
                    final_value = max(values)
                elif func == 'min':
                    final_value = min(values)
                elif func == 'count':
                    final_value = sum(values)
                elif func == 'mean':
                    final_value = sum(values) / len(values)
                else:
                    final_value = values[0]
                
                final_data[result_col].append(final_value)
        
        return DataFrame(final_data)
    
    @staticmethod
    def count_chunks(chunk_reader):
        """
        Count total number of rows across all chunks
        """
        total = 0
        for chunk in chunk_reader:
            total += len(chunk)
        return total
    
    @staticmethod
    def collect_chunks(chunk_reader, max_rows=None):
        """
        Combine all chunks into a single DataFrame
        (Use only if the full dataset fits into memory)
        """
        all_data = None
        total_rows = 0
        
        for chunk in chunk_reader:
            if all_data is None:
                all_data = {col: [] for col in chunk.columns}
            
            if max_rows and total_rows >= max_rows:
                break

            rows_to_add = len(chunk)
            if max_rows and total_rows + rows_to_add > max_rows:
                rows_to_add = max_rows - total_rows

            for col in chunk.columns:
                all_data[col].extend(chunk.data[col][:rows_to_add])
            
            total_rows += rows_to_add
        
        if all_data:
            return DataFrame(all_data)
        
        return DataFrame({})


In [3]:
reader = CSVChunkReader("../data/spotify_songs.csv", chunk_size=1000)
for chunk in reader:
    print(chunk.shape())
    break  # 只讀取第一個塊以測試

(1000, 23)


In [4]:
reader = read_csv_chunks("../data/spotify_songs.csv", chunk_size=1000)
# 只收集前 5000 行
df = ChunkProcessor.collect_chunks(reader, max_rows=5000)
df


track_id             | track_name           | track_artist     | track_popularity | track_album_id       | track_album_name     | track_album_release_date | playlist_name | playlist_id          | playlist_genre | playlist_subgenre | danceability | energy | key | loudness | mode | speechiness | acousticness | instrumentalness | liveness | valence | tempo   | duration_ms
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
6f807x0ima9a1j3VP... | I Don't Care (wit... | Ed Sheeran       | 66               | 2oCs0DGTsRO98Gh5Z... | I Don't Care (wit... | 2019-06-14           | Pop Remix     | 37i9dQZF1DXcZDD7c... | pop            | dance pop         | 0.748        | 0.916  | 6  

In [7]:
df[['playlist_name', 'playlist_genre', 'track_popularity']]

playlist_name | playlist_genre | track_popularity
-------------------------------------------------
Pop Remix     | pop            | 66              
Pop Remix     | pop            | 67              
Pop Remix     | pop            | 70              
Pop Remix     | pop            | 60              
Pop Remix     | pop            | 69              
Pop Remix     | pop            | 67              
Pop Remix     | pop            | 62              
Pop Remix     | pop            | 69              
Pop Remix     | pop            | 68              
Pop Remix     | pop            | 67              

... 4990 more rows

Shape: (5000 rows, 3 columns)